# Manual Input Verification

Displays the four `AircraftCommand` channels (`n_z`, `n_y`, `roll_rate`, `throttle`)
in real time from a physical joystick.

**All input logic runs in C++.** This notebook launches `joystick_verify` as a subprocess
and reads the JSON lines it streams to stdout. No axis pipeline, dead zone, calibration,
trim, or command limiting is implemented here. All display logic is in
[`tools/manual_input_monitor.py`](tools/manual_input_monitor.py).

Design authority: [`docs/architecture/manual_input.md`](../docs/architecture/manual_input.md),
section *Manual Input Monitor*.

---

## Prerequisites

1. Build the project (`mingw32-make -C build`) so `build/tools/joystick_verify.exe` exists.
2. Connect a joystick before running the configuration cell.

---

## Imports

In [ ]:
%matplotlib widget

import json
import queue
import subprocess
import sys
import threading
from pathlib import Path

sys.path.insert(0, str(Path("tools").resolve()))

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from manual_input_monitor import (
    InputMonitorConfig,
    InputMonitorFigure,
    _subprocess_env,
    build_args,
    enumerate_devices,
    stdout_reader,
)

---
## Configuration

Edit these values to match your build location and device.
All axis mapping, dead zone, and calibration options are passed as flags to
`joystick_verify` and handled in C++.

In [ ]:
from pathlib import Path

config = InputMonitorConfig(
    joystick_verify=Path("../build/tools/joystick_verify.exe"),
    config_file=str(Path("gx12_config.json").resolve()),
    rate_hz=50.0,
)

---
## Device Enumeration

Run this cell to launch `joystick_verify` briefly and list detected joystick devices.
Adjust `config.device_name` or `config.device_index` above before opening the display.

In [ ]:
devices = enumerate_devices(config)
print(f"Detected {len(devices)} joystick device(s):")
for d in devices:
    selected = (
        d["index"] == config.device_index and not config.device_name
        or config.device_name.lower() in d["name"].lower()
    )
    marker = " <-- selected" if selected else ""
    print(f"  [{d['index']}]  {d['name']}  ({d['axes']} axes){marker}")
if not devices:
    print("  No devices found. Connect a joystick and re-run this cell.")

---
## Live Monitor

Run this cell to display the live monitor inline. `joystick_verify` is launched as a
background subprocess. A daemon thread reads JSON frames into a queue; `FuncAnimation`
at 20 Hz drains the queue and redraws the display.

**Interrupt the cell** (■ stop button or Kernel → Interrupt) to stop the monitor and
terminate `joystick_verify`. Re-run to restart.

In [ ]:
from manual_input_monitor import _subprocess_env

_q: queue.Queue = queue.Queue()
_ready = threading.Event()
_proc = subprocess.Popen(
    build_args(config),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env=_subprocess_env(),
)
_reader = threading.Thread(target=stdout_reader, args=(_proc, _q, _ready), daemon=True)
_reader.start()

if not _ready.wait(timeout=5.0):
    _proc.terminate()
    raise RuntimeError("joystick_verify did not become ready within 5 s.")

# Use pyplot.figure() so ipympl registers the figure as a widget in this cell.
_fig = plt.figure(figsize=(10, 5.5))
_monitor = InputMonitorFigure(config, fig=_fig)

_last_frame = {
    "n_z": 1.0,
    "n_y": 0.0,
    "roll_rate_wind_rps": 0.0,
    "throttle_nd": config.idle_throttle_nd,
    "connected": False,
}


def _update(_frame_index):
    global _last_frame
    frame = None
    while True:
        try:
            line = _q.get_nowait()
            frame = json.loads(line)
        except queue.Empty:
            break
        except json.JSONDecodeError:
            continue
    if frame is not None:
        _last_frame = frame
    _monitor.redraw(_last_frame)
    if _proc.poll() is not None:
        _anim.event_source.stop()


# Assign to _anim to prevent garbage collection from stopping the animation.
_anim = FuncAnimation(_fig, _update, interval=50, blit=False, cache_frame_data=False)
plt.show()